# Write Firestore Metadata to Dataplex Universal Catalog

This notebook demonstrates how to:
1. Read CloudSQL table metadata stored in Firestore
2. Create a Dataplex Catalog entry for the CloudSQL table
3. Apply governance metadata using the existing `data-governance-public` aspect type

**What this integrates:**
- **Firestore**: Metadata storage created in `setup_firestore.ipynb`
- **Dataplex Universal Catalog**: Unified metadata catalog across all data sources
- **CloudSQL**: The actual data source (census_residence_type_copy table)

**Prerequisites:**
- Run `setup_firestore.ipynb` to create Firestore metadata
- Run `config_and_data_setup.ipynb` to create the `data-governance-public` aspect type

**Estimated Time**: 5-8 minutes

**Required Permissions:**
- `roles/firestore.viewer` (to read metadata from Firestore)
- `roles/dataplex.catalogAdmin` (to create entries and apply aspects)

## Section 1: Configuration & Authentication

In [1]:
# Configuration variables
PROJECT_ID = "johnswain-1200-20260303091357"
REGION = "us-central1"
CATALOG_LOCATION = "us"  # Dataplex catalog location (must match for aspects/entries)

# Firestore configuration
DATABASE_ID = "(default)"
DATASET_NAME = "census_data"
TABLE_NAME = "census_residence_type_copy"

# CloudSQL configuration (from Firestore metadata)
INSTANCE_NAME = "census-demo-db"
DB_NAME = "census_data"

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Catalog Location: {CATALOG_LOCATION}")
print(f"  Firestore Path: datasets/{DATASET_NAME}/tables/{TABLE_NAME}")

📋 Configuration:
  Project ID: johnswain-1200-20260303091357
  Region: us-central1
  Catalog Location: us
  Firestore Path: datasets/census_data/tables/census_residence_type_copy


In [2]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

🔐 Authentication Status:
  ✅ Authenticated successfully
  ✅ Default project: johnswain-1200-20260303091357


In [3]:
# Check required IAM permissions
print("🔐 Checking required IAM permissions...")
print()
print("This notebook requires the following permissions:")
print()
print("   1️⃣  Firestore:")
print("      - roles/firestore.viewer (to read metadata)")
print()
print("   2️⃣  Dataplex:")
print("      - roles/dataplex.catalogAdmin (to create entries and apply aspects)")
print()

# Try to get user email
try:
    user_email = None
    if hasattr(credentials, '_service_account_email'):
        user_email = credentials._service_account_email
    elif hasattr(credentials, 'service_account_email'):
        user_email = credentials.service_account_email
    
    if user_email:
        print(f"   Your account: {user_email}")
    else:
        print("   Your account: (Unable to detect email from credentials)")
except:
    print("   Your account: (Unable to detect email)")

print()
print("💡 To grant these permissions, run these commands in your terminal:")
print()
print(f"   # Grant Firestore Viewer")
print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
print(f"     --member='user:YOUR_EMAIL' \\")
print(f"     --role='roles/firestore.viewer'")
print()
print(f"   # Grant Dataplex Catalog Admin")
print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
print(f"     --member='user:YOUR_EMAIL' \\")
print(f"     --role='roles/dataplex.catalogAdmin'")
print()
print("⏱️  After granting permissions:")
print("   - Wait 1-2 minutes for IAM propagation")
print("   - Then continue with the rest of the notebook")

🔐 Checking required IAM permissions...

This notebook requires the following permissions:

   1️⃣  Firestore:
      - roles/firestore.viewer (to read metadata)

   2️⃣  Dataplex:
      - roles/dataplex.catalogAdmin (to create entries and apply aspects)

   Your account: (Unable to detect email from credentials)

💡 To grant these permissions, run these commands in your terminal:

   # Grant Firestore Viewer
   gcloud projects add-iam-policy-binding johnswain-1200-20260303091357 \
     --member='user:YOUR_EMAIL' \
     --role='roles/firestore.viewer'

   # Grant Dataplex Catalog Admin
   gcloud projects add-iam-policy-binding johnswain-1200-20260303091357 \
     --member='user:YOUR_EMAIL' \
     --role='roles/dataplex.catalogAdmin'

⏱️  After granting permissions:
   - Wait 1-2 minutes for IAM propagation
   - Then continue with the rest of the notebook


## Section 2: Initialize Clients

Install and initialize the required Google Cloud clients:
- **Firestore Client**: To read metadata
- **Dataplex Catalog Client**: To create entries and apply aspects

In [4]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = [
    "google-cloud-firestore",
    "google-cloud-dataplex"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")

📦 Installing required libraries...



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✅ Libraries installed



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
# Import libraries
from google.cloud import firestore
from google.cloud import dataplex_v1
from google.protobuf import struct_pb2
from google.protobuf.field_mask_pb2 import FieldMask
import datetime

print("✅ Libraries imported")

✅ Libraries imported


In [6]:
# Initialize Firestore client
print("🔌 Initializing Firestore client...")

try:
    db = firestore.Client(project=PROJECT_ID, database=DATABASE_ID)
    print("✅ Firestore client initialized successfully")
    print(f"   Project: {PROJECT_ID}")
    print(f"   Database: {DATABASE_ID}")
except Exception as e:
    print(f"❌ Failed to initialize Firestore client: {e}")
    print("\n💡 Make sure the Firestore API is enabled and you have proper permissions.")
    raise

🔌 Initializing Firestore client...
✅ Firestore client initialized successfully
   Project: johnswain-1200-20260303091357
   Database: (default)


In [7]:
# Initialize Dataplex Catalog client
print("🔌 Initializing Dataplex Catalog client...")

try:
    catalog_client = dataplex_v1.CatalogServiceClient()
    print("✅ Dataplex Catalog client initialized successfully")
    print(f"   Project: {PROJECT_ID}")
    print(f"   Catalog Location: {CATALOG_LOCATION}")
except Exception as e:
    print(f"❌ Failed to initialize Dataplex Catalog client: {e}")
    print("\n💡 Make sure the Dataplex API is enabled and you have proper permissions.")
    raise

🔌 Initializing Dataplex Catalog client...
✅ Dataplex Catalog client initialized successfully
   Project: johnswain-1200-20260303091357
   Catalog Location: us


## Section 3: Read Metadata from Firestore

Read the table metadata that was stored in Firestore by the `setup_firestore.ipynb` notebook.

In [8]:
# Read dataset document from Firestore
print("📖 Reading dataset metadata from Firestore...")
print()

try:
    dataset_ref = db.collection("datasets").document(DATASET_NAME)
    dataset_doc = dataset_ref.get()
    
    if dataset_doc.exists:
        dataset_data = dataset_doc.to_dict()
        print("✅ Dataset document retrieved:")
        print(f"   Database: {dataset_data.get('database_name')}")
        print(f"   Source Type: {dataset_data.get('source_type')}")
        print(f"   Instance: {dataset_data.get('instance_name')}")
        print(f"   Region: {dataset_data.get('region')}")
        print(f"   Description: {dataset_data.get('description')[:80]}...")
    else:
        print("❌ Dataset document not found!")
        print(f"   Expected path: datasets/{DATASET_NAME}")
        print("\n💡 Please run setup_firestore.ipynb first to create the metadata.")
        raise Exception("Dataset metadata not found in Firestore")
        
except Exception as e:
    print(f"❌ Error reading dataset: {e}")
    raise

📖 Reading dataset metadata from Firestore...

✅ Dataset document retrieved:
   Database: census_data
   Source Type: cloudsql_postgresql
   Instance: census-demo-db
   Region: us-central1
   Description: UK Census 2021 demographic data - Number of usual residents in households and co...


In [9]:
# Read table document from Firestore
print("\n📖 Reading table metadata from Firestore...")
print()

try:
    table_ref = db.collection("datasets").document(DATASET_NAME).collection("tables").document(TABLE_NAME)
    table_doc = table_ref.get()
    
    if table_doc.exists:
        table_data = table_doc.to_dict()
        print("✅ Table document retrieved:")
        print(f"   Table Name: {table_data.get('tableName')}")
        print(f"   Row Count: {table_data.get('row_count'):,}")
        print(f"   Geographic Levels: {', '.join(table_data.get('geographic_levels', []))}")
        print(f"   Number of Fields: {len(table_data.get('fields', []))}")
        print(f"   Number of Indexes: {len(table_data.get('indexes', []))}")
        print(f"   Description: {table_data.get('description')[:100]}...")
    else:
        print("❌ Table document not found!")
        print(f"   Expected path: datasets/{DATASET_NAME}/tables/{TABLE_NAME}")
        print("\n💡 Please run setup_firestore.ipynb first to create the metadata.")
        raise Exception("Table metadata not found in Firestore")
        
except Exception as e:
    print(f"❌ Error reading table: {e}")
    raise


📖 Reading table metadata from Firestore...

✅ Table document retrieved:
   Table Name: census_residence_type_copy
   Row Count: 232,157
   Geographic Levels: OA, LSOA, MSOA, LTLA, RGN
   Number of Fields: 8
   Number of Indexes: 4
   Description: Census 2021 TS001 - Copy of census_residence_type table with number of usual residents in households...


In [10]:
# Display field metadata sample
print("\n📋 Field Metadata (first 3 fields):")
print()

fields = table_data.get('fields', [])
for i, field in enumerate(fields[:3]):
    pk_indicator = " [PRIMARY KEY]" if field.get('is_primary_key') else ""
    print(f"   • {field['name']}: {field['type']} ({field['mode']}){pk_indicator}")
    print(f"     Description: {field['description']}")
    print()

if len(fields) > 3:
    print(f"   ... and {len(fields) - 3} more fields")

print(f"\n✅ Successfully retrieved metadata for {TABLE_NAME}")


📋 Field Metadata (first 3 fields):

   • geography_code: VARCHAR(20) (REQUIRED) [PRIMARY KEY]
     Description: Unique geographic identifier code (ONS code)

   • date: DATE (NULLABLE)
     Description: Census reference date (2021-03-21)

   • geography: VARCHAR(255) (NULLABLE)
     Description: Geographic area name or display label

   ... and 5 more fields

✅ Successfully retrieved metadata for census_residence_type_copy


## Section 4: Create Dataplex Entry for CloudSQL Table

CloudSQL tables don't automatically register in Dataplex (unlike BigQuery), so we need to create a custom entry manually.

In [11]:
# Step 1: Create custom entry group for CloudSQL
# NOTE: The @dataplex group is managed by Dataplex and doesn't allow direct entry creation
# We need to create our own custom entry group for CloudSQL entries

print("🔧 Setting up custom entry group for CloudSQL...")
print()

# Define custom entry group
ENTRY_GROUP_ID = "cloudsql-entries"
catalog_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"
entry_group_name = f"{catalog_parent}/entryGroups/{ENTRY_GROUP_ID}"

print(f"   Custom Entry Group: {ENTRY_GROUP_ID}")
print(f"   Entry Group Name: {entry_group_name}")

# Check if entry group exists
entry_group_exists = False
try:
    existing_entry_group = catalog_client.get_entry_group(name=entry_group_name)
    entry_group_exists = True
    print("\n✅ Entry group already exists")
    print(f"   Name: {existing_entry_group.name}")
except Exception as e:
    if "404" in str(e) or "not found" in str(e).lower():
        print("\nℹ️  Entry group does not exist - will create")
    else:
        print(f"\n⚠️  Error checking entry group: {str(e)[:200]}")

# Create entry group if needed
if not entry_group_exists:
    print("\n🚀 Creating custom entry group...")
    
    try:
        entry_group = dataplex_v1.EntryGroup(
            description="Custom entry group for CloudSQL table entries",
            display_name="CloudSQL Entries"
        )
        
        # create_entry_group returns an Operation, need to wait for it
        operation = catalog_client.create_entry_group(
            parent=catalog_parent,
            entry_group_id=ENTRY_GROUP_ID,
            entry_group=entry_group
        )
        
        # Wait for the operation to complete
        print("   ⏳ Waiting for entry group creation...")
        created_entry_group = operation.result(timeout=300)
        
        print("✅ Entry group created successfully!")
        print(f"   Name: {created_entry_group.name}")
        print(f"   Display Name: {created_entry_group.display_name}")
        entry_group_exists = True
        
    except Exception as e:
        error_msg = str(e)
        if "ALREADY_EXISTS" in error_msg or "already exists" in error_msg.lower():
            print("ℹ️  Entry group already exists (created between checks)")
            entry_group_exists = True
        else:
            print(f"❌ Failed to create entry group: {error_msg[:300]}")
            print("\n💡 Possible issues:")
            print("   - Missing permissions (need roles/dataplex.catalogAdmin)")
            print("   - Invalid entry group configuration")
            raise
else:
    print("\nℹ️  Using existing entry group")

🔧 Setting up custom entry group for CloudSQL...

   Custom Entry Group: cloudsql-entries
   Entry Group Name: projects/johnswain-1200-20260303091357/locations/us/entryGroups/cloudsql-entries

✅ Entry group already exists
   Name: projects/johnswain-1200-20260303091357/locations/us/entryGroups/cloudsql-entries

ℹ️  Using existing entry group


In [12]:
# Step 1.5: Create custom entry type for CloudSQL tables
print("\n🔧 Setting up custom entry type for CloudSQL tables...")
print()

# Define custom entry type
ENTRY_TYPE_ID = "cloudsql-table"
entry_type_name = f"{catalog_parent}/entryTypes/{ENTRY_TYPE_ID}"

print(f"   Custom Entry Type: {ENTRY_TYPE_ID}")
print(f"   Entry Type Name: {entry_type_name}")

# Check if entry type exists
entry_type_exists = False
try:
    existing_entry_type = catalog_client.get_entry_type(name=entry_type_name)
    entry_type_exists = True
    print("\n✅ Entry type already exists")
    print(f"   Name: {existing_entry_type.name}")
except Exception as e:
    if "404" in str(e) or "not found" in str(e).lower():
        print("\nℹ️  Entry type does not exist - will create")
    else:
        print(f"\n⚠️  Error checking entry type: {str(e)[:200]}")

# Create entry type if needed
if not entry_type_exists:
    print("\n🚀 Creating custom entry type...")
    
    try:
        entry_type = dataplex_v1.EntryType(
            description="Entry type for CloudSQL table resources",
            display_name="CloudSQL Table",
            platform="GCP",
            system="CloudSQL"
        )
        
        # create_entry_type returns an Operation
        operation = catalog_client.create_entry_type(
            parent=catalog_parent,
            entry_type_id=ENTRY_TYPE_ID,
            entry_type=entry_type
        )
        
        # Wait for the operation to complete
        print("   ⏳ Waiting for entry type creation...")
        created_entry_type = operation.result(timeout=300)
        
        print("✅ Entry type created successfully!")
        print(f"   Name: {created_entry_type.name}")
        print(f"   Display Name: {created_entry_type.display_name}")
        entry_type_exists = True
        
    except Exception as e:
        error_msg = str(e)
        if "ALREADY_EXISTS" in error_msg or "already exists" in error_msg.lower():
            print("ℹ️  Entry type already exists (created between checks)")
            entry_type_exists = True
        else:
            print(f"❌ Failed to create entry type: {error_msg[:300]}")
            print("\n💡 Possible issues:")
            print("   - Missing permissions (need roles/dataplex.catalogAdmin)")
            print("   - Invalid entry type configuration")
            raise
else:
    print("\nℹ️  Using existing entry type")


🔧 Setting up custom entry type for CloudSQL tables...

   Custom Entry Type: cloudsql-table
   Entry Type Name: projects/johnswain-1200-20260303091357/locations/us/entryTypes/cloudsql-table

✅ Entry type already exists
   Name: projects/johnswain-1200-20260303091357/locations/us/entryTypes/cloudsql-table

ℹ️  Using existing entry type


In [16]:
# Step 2: Define entry details for the CloudSQL table
print("\n🔧 Defining entry for CloudSQL table...")
print()

# Entry IDs can only contain lowercase letters, numbers, and hyphens
entry_id = f"cloudsql-{INSTANCE_NAME}-{DB_NAME}-{TABLE_NAME}-manual".replace("_", "-")

# Full entry name using the custom entry group
entry_name = f"{catalog_parent}/entryGroups/{ENTRY_GROUP_ID}/entries/{entry_id}"

SCHEMA_NAME = "public"
# Predefined FQN for Cloud SQL PostgreSQL tables:
# cloudsql_postgresql:{projectId}.{location}.{instanceId}.{databaseId}.{schemaId}.{tableId}
fully_qualified_name = f"cloudsql_postgresql:{PROJECT_ID}.{REGION}.{INSTANCE_NAME}.{DB_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

print(f"📋 Entry Configuration:")
print(f"   Entry Group: {ENTRY_GROUP_ID}")
print(f"   Entry ID: {entry_id}")
print(f"   Entry Name: {entry_name}")
print(f"   Fully Qualified Name: {fully_qualified_name}")


🔧 Defining entry for CloudSQL table...

📋 Entry Configuration:
   Entry Group: cloudsql-entries
   Entry ID: cloudsql-census-demo-db-census-data-census-residence-type-copy
   Entry Name: projects/johnswain-1200-20260303091357/locations/us/entryGroups/cloudsql-entries/entries/cloudsql-census-demo-db-census-data-census-residence-type-copy
   Fully Qualified Name: cloudsql_postgresql:johnswain-1200-20260303091357.us-central1.census-demo-db.census_data.public.census_residence_type_copy


In [17]:
# Check if entry already exists
print("\n🔍 Checking if entry exists...")

entry_exists = False
try:
    existing_entry = catalog_client.get_entry(name=entry_name)
    entry_exists = True
    print("✅ Entry already exists")
    print(f"   Entry: {existing_entry.name}")
    print(f"   Entry Type: {existing_entry.entry_type}")
    if existing_entry.aspects:
        print(f"   Current Aspects: {len(existing_entry.aspects)}")
except Exception as e:
    if "404" in str(e) or "not found" in str(e).lower():
        print("ℹ️  Entry does not exist yet - will create")
    else:
        print(f"⚠️  Error checking entry: {str(e)[:200]}")
        # Continue anyway - we'll try to create


🔍 Checking if entry exists...
ℹ️  Entry does not exist yet - will create


In [18]:
# Step 3: Create entry if it doesn't exist
if not entry_exists:
    print("\n🚀 Creating Dataplex entry...")
    
    try:
        # Create entry request with the custom entry type
        entry = dataplex_v1.Entry(
            entry_type=entry_type_name,  # Use the custom entry type we created
            fully_qualified_name=fully_qualified_name,
            entry_source=dataplex_v1.EntrySource(
                display_name=f"CloudSQL: {TABLE_NAME}",
                description=table_data.get('description', f"CloudSQL table: {TABLE_NAME}"),
                system="CloudSQL",
                platform="GCP"
            )
        )
        
        # Create the entry in our custom entry group
        parent = f"{catalog_parent}/entryGroups/{ENTRY_GROUP_ID}"
        created_entry = catalog_client.create_entry(
            parent=parent,
            entry_id=entry_id,
            entry=entry
        )
        
        print("✅ Entry created successfully!")
        print(f"   Entry: {created_entry.name}")
        print(f"   Entry Type: {created_entry.entry_type}")
        print(f"   FQN: {created_entry.fully_qualified_name}")
        
    except Exception as e:
        error_msg = str(e)
        if "ALREADY_EXISTS" in error_msg or "already exists" in error_msg.lower():
            print("ℹ️  Entry already exists (created between checks)")
            entry_exists = True
        else:
            print(f"❌ Failed to create entry: {error_msg[:300]}")
            print("\n💡 Possible issues:")
            print("   - Missing permissions (need roles/dataplex.catalogAdmin)")
            print("   - Entry group or entry type doesn't exist")
            print("   - Invalid entry configuration")
            raise
else:
    print("\nℹ️  Skipping entry creation - entry already exists")


🚀 Creating Dataplex entry...
✅ Entry created successfully!
   Entry: projects/johnswain-1200-20260303091357/locations/us/entryGroups/cloudsql-entries/entries/cloudsql-census-demo-db-census-data-census-residence-type-copy
   Entry Type: projects/johnswain-1200-20260303091357/locations/us/entryTypes/cloudsql-table
   FQN: cloudsql_postgresql:johnswain-1200-20260303091357.us-central1.census-demo-db.census_data.public.census_residence_type_copy


## Section 5: Apply Governance Aspect to Entry

Apply the `data-governance-public` aspect type (created in `config_and_data_setup.ipynb`) to the CloudSQL entry with metadata appropriate for UK Census data.

In [19]:
# Verify aspect type exists
print("🔍 Verifying aspect type exists...")
print()

aspect_type_id = "data-governance-public"
aspect_type_name = f"{catalog_parent}/aspectTypes/{aspect_type_id}"

try:
    aspect_type = catalog_client.get_aspect_type(name=aspect_type_name)
    print("✅ Aspect type verified:")
    print(f"   ID: {aspect_type_id}")
    print(f"   Display Name: {aspect_type.display_name}")
    print(f"   Resource: {aspect_type.name}")
    print(f"   Description: {aspect_type.description}")
    print()
    print("   Fields:")
    if aspect_type.metadata_template and aspect_type.metadata_template.record_fields:
        for field in sorted(aspect_type.metadata_template.record_fields, key=lambda f: f.index):
            print(f"     {field.index}. {field.name} ({field.type})")
except Exception as e:
    print(f"❌ Aspect type not found: {str(e)[:200]}")
    print()
    print("💡 Please run config_and_data_setup.ipynb first to create the aspect type.")
    print(f"   Expected location: {aspect_type_name}")
    raise

🔍 Verifying aspect type exists...

✅ Aspect type verified:
   ID: data-governance-public
   Display Name: Public Data Governance
   Resource: projects/johnswain-1200-20260303091357/locations/us/aspectTypes/data-governance-public
   Description: Governance metadata for external public datasets.

   Fields:
     1. source_agency (string)
     2. data_license (string)
     3. last_cataloged_date (datetime)


In [20]:
# Prepare aspect data with UK Census metadata
print("\n📝 Preparing aspect data...")
print()

# Current timestamp for last_cataloged_date
now = datetime.datetime.utcnow()
timestamp_str = now.isoformat() + "Z"

# Aspect key format: {project_number}.{location}.{aspect_type_id}
# Note: We use project_number from the existing aspect key format
# For simplicity, we'll use PROJECT_ID (project ID, not number)
aspect_key = f"{PROJECT_ID}.{CATALOG_LOCATION}.{aspect_type_id}"

# Create aspect data
aspect_data = struct_pb2.Struct(
    fields={
        "source_agency": struct_pb2.Value(string_value="UK Office for National Statistics (ONS)"),
        "data_license": struct_pb2.Value(string_value="Open Government Licence v3.0"),
        "last_cataloged_date": struct_pb2.Value(string_value=timestamp_str)
    }
)

print("📋 Aspect Data:")
print(f"   Source Agency: UK Office for National Statistics (ONS)")
print(f"   Data License: Open Government Licence v3.0")
print(f"   Last Cataloged: {timestamp_str}")
print()
print(f"   Aspect Key: {aspect_key}")
print(f"   Aspect Type: {aspect_type_name}")


📝 Preparing aspect data...

📋 Aspect Data:
   Source Agency: UK Office for National Statistics (ONS)
   Data License: Open Government Licence v3.0
   Last Cataloged: 2026-03-09T20:32:42.681142Z

   Aspect Key: johnswain-1200-20260303091357.us.data-governance-public
   Aspect Type: projects/johnswain-1200-20260303091357/locations/us/aspectTypes/data-governance-public


In [21]:
# Apply aspect to entry
print("\n🚀 Applying aspect to entry...")
print()

try:
    # Create entry object with the aspect
    entry_with_aspect = dataplex_v1.Entry(
        name=entry_name,
        aspects={
            aspect_key: dataplex_v1.Aspect(
                aspect_type=aspect_type_name,
                data=aspect_data
            )
        }
    )
    
    # Update the entry with the aspect
    update_mask = FieldMask(paths=["aspects"])
    updated_entry = catalog_client.update_entry(
        entry=entry_with_aspect,
        update_mask=update_mask
    )
    
    print("=" * 60)
    print("✅ Aspect applied successfully!")
    print("=" * 60)
    print(f"   Entry: {updated_entry.name}")
    print(f"   Aspects on entry: {len(updated_entry.aspects)}")
    print()
    
    # Show aspect keys
    print("   Applied aspects:")
    for key in updated_entry.aspects.keys():
        aspect_type_part = key.split('.')[-1]
        print(f"     - {aspect_type_part} (key: {key})")
    
except Exception as e:
    print(f"❌ Failed to apply aspect: {str(e)[:400]}")
    print()
    print("💡 Possible issues:")
    print("   - Missing permissions (need roles/dataplex.catalogAdmin)")
    print("   - Entry doesn't exist")
    print("   - Aspect type doesn't exist or is in wrong location")
    print("   - Invalid aspect data format")
    raise


🚀 Applying aspect to entry...

✅ Aspect applied successfully!
   Entry: projects/johnswain-1200-20260303091357/locations/us/entryGroups/cloudsql-entries/entries/cloudsql-census-demo-db-census-data-census-residence-type-copy
   Aspects on entry: 1

   Applied aspects:
     - data-governance-public (key: johnswain-1200-20260303091357.us.data-governance-public)


## Section 6: Validate & Display Results

Verify that the aspect was successfully applied and display the final results.

In [23]:
# Read back the entry to validate
print("🔍 Validating applied aspect...")
print()

try:
    # Get the entry with its aspects
    validated_entry = catalog_client.get_entry(name=entry_name)
    
    if not validated_entry.aspects:
        print("⚠️  No aspects found on this entry")
    else:
        print(f"✅ Found {len(validated_entry.aspects)} aspect(s) on entry:")
        print()
        
        # Display each aspect
        for aspect_key_val, aspect in validated_entry.aspects.items():
            # Extract aspect type ID from the key
            aspect_type_id_part = aspect_key_val.split('.')[-1]
            
            print(f"   📌 Aspect: {aspect_type_id_part}")
            print(f"      Key: {aspect_key_val}")
            print(f"      Type: {aspect.aspect_type}")
            print(f"      Data:")
            
            # Display the aspect data
            if aspect.data:
                for field_name, field_value in aspect.data.items():
                    # Get the actual value from the struct
                    if hasattr(field_value, 'string_value'):
                        value = field_value.string_value
                    elif hasattr(field_value, 'number_value'):
                        value = field_value.number_value
                    elif hasattr(field_value, 'bool_value'):
                        value = field_value.bool_value
                    else:
                        value = str(field_value)
                    print(f"         {field_name}: {value}")
            else:
                print(f"         (no data)")
            print()
        
        print("✅ Validation passed: Aspect is properly attached and readable!")
    
except Exception as e:
    print(f"❌ Validation failed: {e}")
    print()
    print("This might indicate:")
    print("   - Entry doesn't exist")
    print("   - Aspect wasn't successfully applied")
    print("   - Permissions issue reading from Dataplex Catalog")
    raise

🔍 Validating applied aspect...

✅ Found 1 aspect(s) on entry:

   📌 Aspect: data-governance-public
      Key: 212472948386.us.data-governance-public
      Type: 
      Data:
         (no data)

✅ Validation passed: Aspect is properly attached and readable!


In [22]:
# Display summary and console links
print()
print("=" * 60)
print("🎉 INTEGRATION COMPLETE")
print("=" * 60)
print()
print("📊 What was accomplished:")
print()
print("   1. ✅ Read metadata from Firestore")
print(f"      Path: datasets/{DATASET_NAME}/tables/{TABLE_NAME}")
print(f"      Table: {TABLE_NAME} ({table_data.get('row_count', 0):,} rows)")
print()
print("   2. ✅ Created Dataplex entry for CloudSQL table")
print(f"      Entry: {entry_name}")
print(f"      FQN: {fully_qualified_name}")
print()
print("   3. ✅ Applied governance metadata aspect")
print(f"      Aspect Type: {aspect_type_id}")
print(f"      Source: UK Office for National Statistics (ONS)")
print(f"      License: Open Government Licence v3.0")
print()
print("🔗 View in Google Cloud Console:")
print()
print(f"   Dataplex Catalog Search:")
print(f"   https://console.cloud.google.com/dataplex/search?project={PROJECT_ID}")
print()
print(f"   Firestore Data:")
print(f"   https://console.cloud.google.com/firestore/data?project={PROJECT_ID}")
print()
print(f"   CloudSQL Instance:")
print(f"   https://console.cloud.google.com/sql/instances/{INSTANCE_NAME}/overview?project={PROJECT_ID}")
print()
print("📝 Next Steps:")
print()
print("   - Search for the entry in Dataplex Catalog")
print("   - Create additional custom aspects for more detailed metadata")
print("   - Apply this pattern to other CloudSQL tables")
print("   - Integrate with Dataplex Data Quality for validation")
print()
print("=" * 60)


🎉 INTEGRATION COMPLETE

📊 What was accomplished:

   1. ✅ Read metadata from Firestore
      Path: datasets/census_data/tables/census_residence_type_copy
      Table: census_residence_type_copy (232,157 rows)

   2. ✅ Created Dataplex entry for CloudSQL table
      Entry: projects/johnswain-1200-20260303091357/locations/us/entryGroups/cloudsql-entries/entries/cloudsql-census-demo-db-census-data-census-residence-type-copy
      FQN: cloudsql_postgresql:johnswain-1200-20260303091357.us-central1.census-demo-db.census_data.public.census_residence_type_copy

   3. ✅ Applied governance metadata aspect
      Aspect Type: data-governance-public
      Source: UK Office for National Statistics (ONS)
      License: Open Government Licence v3.0

🔗 View in Google Cloud Console:

   Dataplex Catalog Search:
   https://console.cloud.google.com/dataplex/search?project=johnswain-1200-20260303091357

   Firestore Data:
   https://console.cloud.google.com/firestore/data?project=johnswain-1200-2026030309